# Комп'ютерна графіка — тренажер «Три проекції»

## Узгоджена система проекцій

Фронтальна, горизонтальна та профільна проекції використовують спільні масштаби координат. Зміна координати точки узгоджено відображається у відповідних проекціях.

Запустіть одну наступну code-комірку.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

COLORS = {"A": "tab:red", "B": "tab:blue", "C": "tab:green"}


def make_points(Ax, Ay, Az, Bx, By, Bz, Cx, Cy, Cz):
    return np.array([
        [Ax, Ay, Az],
        [Bx, By, Bz],
        [Cx, Cy, Cz]
    ], dtype=float)


def plane_type(P):
    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) < 1e-9:
        return "⚠️ Площина не визначена: A, B, C колінеарні"

    if np.allclose(P[:, 2], P[0, 2]):
        return "ГОРИЗОНТАЛЬНА ПЛОЩИНА  →  Z = const"

    if np.allclose(P[:, 1], P[0, 1]):
        return "ФРОНТАЛЬНА ПЛОЩИНА  →  Y = const"

    if np.allclose(P[:, 0], P[0, 0]):
        return "ПРОФІЛЬНА ПЛОЩИНА  →  X = const"

    return "ПЛОЩИНА ЗАГАЛЬНОГО ПОЛОЖЕННЯ"


def common_limits(P):
    low = P.min(axis=0)
    high = P.max(axis=0)

    span = np.maximum(high - low, 2)
    margin = 0.25 * span

    return low - margin, high + margin


def draw_projection(ax, P, first, second, title, limits, show_lines):
    names = ["X", "Y", "Z"]
    low, high = limits

    Q = P[:, [first, second]]

    ax.set_xlim(low[first], high[first])
    ax.set_ylim(low[second], high[second])
    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel(names[first])
    ax.set_ylabel(names[second])
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.25)

    contour = [0, 1, 2, 0]

    ax.fill(
        Q[:, 0],
        Q[:, 1],
        alpha=0.12
    )

    ax.plot(
        Q[contour, 0],
        Q[contour, 1],
        color="black",
        linewidth=1.5
    )

    for i, name in enumerate(["A", "B", "C"]):
        x, y = Q[i]

        ax.scatter(
            x,
            y,
            s=80,
            color=COLORS[name],
            zorder=5
        )

        ax.annotate(
            name,
            (x, y),
            xytext=(8, 8),
            textcoords="offset points",
            fontweight="bold"
        )

        if show_lines:
            ax.axvline(
                x,
                color=COLORS[name],
                linestyle="--",
                alpha=0.22
            )

            ax.axhline(
                y,
                color=COLORS[name],
                linestyle="--",
                alpha=0.22
            )


def draw_3d(ax, P, limits):
    low, high = limits

    ax.set_xlim(low[0], high[0])
    ax.set_ylim(low[1], high[1])
    ax.set_zlim(low[2], high[2])

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("Допоміжний 3D-вигляд", fontweight="bold")

    contour = [0, 1, 2, 0]

    ax.plot(
        P[contour, 0],
        P[contour, 1],
        P[contour, 2],
        color="black",
        linewidth=1.5
    )

    normal = np.cross(P[1] - P[0], P[2] - P[0])

    if np.linalg.norm(normal) > 1e-9:
        ax.plot_trisurf(
            P[:, 0],
            P[:, 1],
            P[:, 2],
            alpha=0.12
        )
    else:
        ax.text2D(
            0.05,
            0.92,
            "Площина не визначена",
            transform=ax.transAxes,
            fontweight="bold"
        )

    for i, name in enumerate(["A", "B", "C"]):
        x, y, z = P[i]

        ax.scatter(
            x,
            y,
            z,
            s=80,
            color=COLORS[name]
        )

        ax.text(
            x,
            y,
            z,
            "  " + name,
            fontweight="bold"
        )

    ax.view_init(elev=24, azim=-58)


def trainer(
    Ax=2, Ay=2, Az=2,
    Bx=8, By=3, Bz=6,
    Cx=5, Cy=8, Cz=4,
    show_lines=True
):
    P = make_points(
        Ax, Ay, Az,
        Bx, By, Bz,
        Cx, Cy, Cz
    )

    limits = common_limits(P)

    fig = plt.figure(figsize=(15, 9))

    ax_front = fig.add_subplot(221)
    ax_profile = fig.add_subplot(222)
    ax_horizontal = fig.add_subplot(223)
    ax_3d = fig.add_subplot(224, projection="3d")

    draw_projection(
        ax_front,
        P,
        0, 2,
        "Фронтальна проекція XZ",
        limits,
        show_lines
    )

    draw_projection(
        ax_profile,
        P,
        1, 2,
        "Профільна проекція YZ",
        limits,
        show_lines
    )

    draw_projection(
        ax_horizontal,
        P,
        0, 1,
        "Горизонтальна проекція XY",
        limits,
        show_lines
    )

    draw_3d(ax_3d, P, limits)

    fig.suptitle(
        plane_type(P),
        fontsize=17,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()


interact(
    trainer,
    Ax=FloatSlider(min=0, max=10, step=1, value=2, description="A: X", continuous_update=False),
    Ay=FloatSlider(min=0, max=10, step=1, value=2, description="A: Y", continuous_update=False),
    Az=FloatSlider(min=0, max=10, step=1, value=2, description="A: Z", continuous_update=False),
    Bx=FloatSlider(min=0, max=10, step=1, value=8, description="B: X", continuous_update=False),
    By=FloatSlider(min=0, max=10, step=1, value=3, description="B: Y", continuous_update=False),
    Bz=FloatSlider(min=0, max=10, step=1, value=6, description="B: Z", continuous_update=False),
    Cx=FloatSlider(min=0, max=10, step=1, value=5, description="C: X", continuous_update=False),
    Cy=FloatSlider(min=0, max=10, step=1, value=8, description="C: Y", continuous_update=False),
    Cz=FloatSlider(min=0, max=10, step=1, value=4, description="C: Z", continuous_update=False),
    show_lines=Checkbox(value=True, description="Показувати проекційні лінії")
)


## Перевірка

- X має однаковий масштаб у фронтальній та горизонтальній проекціях.
- Y має однаковий масштаб у горизонтальній та профільній.
- Z має однаковий масштаб у фронтальній та профільній.
- Точки A, B, C мають узгоджене положення у всіх проекціях.

Тепер це три графічні представлення однієї просторової моделі, а не три незалежні графіки.